# Chapter 06 답안 양식. 데이터를 보며 질문을 만드는 EDA

## 제출 정보
- 이름: Jongeun Kim
- GitHub ID: risa0122
- 작성일: 2026-10-21
- 최종 Notebook URL: https://github.com/risa0122/llm-data-analysis-study/blob/main/chapter06/chapter06.ipynb

## 1. 내가 정한 EDA 질문

| 질문 | 분석 범위 | 지표 | 검증 방법 |
| --- | --- | --- | --- |
| 카테고리별 completed 주문 금액과 수량은 어떻게 다른가? | processed 주문 상세 중 주문표의 completed 상태와 연결된 행. 상품 미연결은 별도 분류 | `line_total` 합계, 수량 합계, 금액 비중, 수량 가중 평균 단가 | `many_to_one` 병합·미매칭 확인, 원본 completed 합계와 대조 |
| 월별 completed 금액 변화는 주문 수와 주문당 평균 금액에서 어떻게 나타나는가? | 위와 같은 상세 행. 날짜가 없으면 월 미확인으로 분리 | 월별 금액, `order_id.nunique()`, 금액÷고유 주문 수 | 월 미확인 금액까지 포함해 총합 검증, 주문 건수 합계 대조 |
| completed 구매가 여러 번인 고객과 한 번인 고객의 금액은 어떻게 다른가? | 위와 같은 상세 행 중 고객표와 연결된 고객. 고객 미연결 금액은 별도 보존 | 고객별 주문 수, 금액, 주문당 평균 금액. 1건/2건 이상 그룹 비교 | 고객별 금액 총합 및 미연결 금액 대조, 반복 구매 그룹의 분모 확인 |

### 왜 이 질문을 선택했는가?

금액이 큰 집단을 찾는 데서 그치지 않고, 수량·주문 수·반복 구매를 함께 확인하고 싶었다. 주문 상세의 수량·단가, 상품 카테고리, 주문 상태·날짜·고객 ID가 있어 계산할 수 있다.

`line_total = quantity × unit_price`로 계산한다. `total_sales`는 확인 가능한 completed 주문 상세의 금액이며 실제 회사 매출액·순매출·이익을 의미하지 않는다. 원가·방문·마케팅 자료가 없어 수익성, 구매 전환율, 마케팅 효과는 답할 수 없다.

![EDA 질문과 분석 기준](images/step01_questions.png)

## 2. 핵심 집계 결과

- completed 원본 합계: 141,340,000
- 카테고리별 합계: 141,340,000 — 카테고리 미확인 4,015,000 포함
- 월별 합계: 141,340,000 — 월 미확인 1,903,000 포함
- 총합 일치 여부: 일치. 아래 코드로 원본·카테고리·월·고객 합계를 대조했다.

Chapter 05에서 저장한 processed CSV 4개를 사용한다. 고객·상품·주문 분포는 각 표 전체를 확인하고, 금액 집계는 completed 상세 446행·179건으로 통일한다. 주문표의 completed 180건 중 1건은 남아 있는 주문 상세가 없어 금액을 계산할 수 없다.

In [27]:
from pathlib import Path
import hashlib
import pandas as pd
import numpy as np
from IPython.display import display

pd.set_option('display.max_columns', 12)
pd.set_option('display.max_colwidth', 90)
pd.set_option('display.float_format', lambda value: f'{value:,.2f}')

current = Path.cwd().resolve()
candidates = [current, current / 'chapter06', current.parent / 'chapter06']
CHAPTER_DIR = next((p for p in candidates if (p / 'chapter06.ipynb').is_file()), None)
if CHAPTER_DIR is None:
    raise FileNotFoundError('chapter06.ipynb가 있는 폴더 또는 저장소 루트에서 실행한다.')
PROJECT_ROOT = CHAPTER_DIR.parent
input_candidates = [CHAPTER_DIR / 'data/processed', PROJECT_ROOT / 'data/processed']
filenames = ['customers_clean.csv', 'products_clean.csv', 'orders_clean.csv', 'order_items_clean.csv']
PROCESSED_DIR = next((p for p in input_candidates if all((p / n).is_file() for n in filenames)), None)
if PROCESSED_DIR is None:
    raise FileNotFoundError('Chapter 05 processed CSV 4개가 필요하다. 저장소 루트에서 전처리 스크립트를 실행한다.')
REPORT_DIR = CHAPTER_DIR / 'reports'
REPORT_DIR.mkdir(exist_ok=True)
print('입력: data/processed의 Chapter 05 전처리 결과 4개')
print('Python 데이터 분석 환경 / pandas:', pd.__version__)

입력: data/processed의 Chapter 05 전처리 결과 4개
Python 데이터 분석 환경 / pandas: 3.0.5


In [28]:
# 입력 버전 확인
expected_hashes = {'customers_clean.csv': 'f14a59af4a8a900c0b9f72c7fab3367df2ba1e1419f24fb750c1f366174d7d12', 'order_items_clean.csv': '82389ef808e4f9db56ad76e52fe511b0face3f016cb817cac073a30d7b46d0d2', 'orders_clean.csv': '8b1faa0ffd2c66313d5de82686e2839509800bfb214a1d0ad80f4ddd92881ecc', 'products_clean.csv': '6a6632baa5bd328f95587797c9d4fd786b1ad15761484dbe9d6321054ad4b9f7'}
input_hashes = {name: hashlib.sha256((PROCESSED_DIR / name).read_bytes()).hexdigest() for name in filenames}
assert input_hashes == expected_hashes, '입력 데이터가 바뀌었다. 수치와 해석을 다시 검토한다.'
customers = pd.read_csv(PROCESSED_DIR / 'customers_clean.csv')
products = pd.read_csv(PROCESSED_DIR / 'products_clean.csv')
orders = pd.read_csv(PROCESSED_DIR / 'orders_clean.csv')
order_items = pd.read_csv(PROCESSED_DIR / 'order_items_clean.csv')
raw_items = order_items.copy(deep=True)
data = {'customers': customers, 'products': products, 'orders': orders, 'order_items': order_items}
key_columns = {'customers': 'customer_id', 'products': 'product_id', 'orders': 'order_id', 'order_items': 'order_item_id'}
structure = pd.DataFrame([
    {'데이터': name, '행': len(df), '열': len(df.columns),
     'key 중복': int(df[key_columns[name]].duplicated().sum()),
     'key 결측': int(df[key_columns[name]].isna().sum()), '컬럼': ', '.join(df.columns)}
    for name, df in data.items()
])
assert structure[['key 중복', 'key 결측']].eq(0).all().all()
display(structure)
print('Chapter 05 processed 파일 체크섬 일치:', input_hashes == expected_hashes)

,데이터,행,열,key 중복,key 결측,컬럼
0,customers,150,6,0,0,"customer_id, name, gender, age, city, signup_date"
1,products,96,4,0,0,"product_id, product_name, category, price"
2,orders,300,7,0,0,"order_id, customer_id, order_date, payment_method, order_status, order_month, order_da..."
3,order_items,752,6,0,0,"order_item_id, order_id, product_id, quantity, unit_price, line_total"


Chapter 05 processed 파일 체크섬 일치: True


In [29]:
# 날짜와 원본 금액의 계산 기준 확인
orders['order_date'] = pd.to_datetime(orders['order_date'], errors='coerce')
customers['signup_date'] = pd.to_datetime(customers['signup_date'], errors='coerce')
orders['order_month'] = orders['order_date'].dt.strftime('%Y-%m').fillna('월 미확인')
for column in ['quantity', 'unit_price', 'line_total']:
    assert pd.api.types.is_numeric_dtype(order_items[column])
    assert np.isfinite(order_items[column]).all()
assert order_items['quantity'].gt(0).all()
assert order_items['quantity'].mod(1).eq(0).all()
assert order_items['unit_price'].ge(0).all()
assert order_items['line_total'].eq(order_items['quantity'] * order_items['unit_price']).all()
print('주문 날짜 미확인:', int(orders['order_date'].isna().sum()), '건')
print('주문 날짜 범위:', orders['order_date'].min().date(), '~', orders['order_date'].max().date())
print('작성일 이후 주문:', int(orders['order_date'].gt(pd.Timestamp('2026-10-21')).sum()), '건')
print('line_total = quantity × unit_price:', True)
display(order_items[['quantity', 'unit_price', 'line_total']].head())

주문 날짜 미확인: 3 건
주문 날짜 범위: 2025-07-01 ~ 2028-12-31
작성일 이후 주문: 2 건
line_total = quantity × unit_price: True


,quantity,unit_price,line_total
0,2.00,"172,000.00","344,000.00"
1,4.00,"79,000.00","316,000.00"
2,3.00,"87,000.00","261,000.00"
3,1.00,"187,000.00","187,000.00"
4,5.00,"125,000.00","625,000.00"


고객·상품·주문 분포는 공식 Notebook의 순서에 따라 확인했다. 나이·상품 가격의 극단값과 카테고리 내부 공백이 남아 있어, 평균이나 범주 차이를 곧바로 정상 특성으로 해석하지 않는다.

In [30]:
customers['age'].describe()

count   150.00
mean     43.91
std      17.80
min      18.00
25%      30.00
50%      41.00
75%      56.75
max     150.00
Name: age, dtype: float64

In [31]:
customer_city = customers['city'].value_counts(dropna=False).reset_index()
customer_city.columns = ['city', 'customer_count']
customer_city['customer_ratio'] = (
    customer_city['customer_count'] / customer_city['customer_count'].sum() * 100
).round(2)

customer_city

,city,customer_count,customer_ratio
0,부산,23,15.33
1,대구,21,14.00
2,광주,21,14.00
3,서울,18,12.00
4,울산,17,11.33
5,인천,16,10.67
6,수원,10,6.67
7,성남,9,6.00
8,고양,8,5.33
9,대전,5,3.33


In [32]:
customer_gender = customers['gender'].value_counts(dropna=False).reset_index()
customer_gender.columns = ['gender', 'customer_count']
customer_gender['customer_ratio'] = (
    customer_gender['customer_count'] / customer_gender['customer_count'].sum() * 100
).round(2)

customer_gender

,gender,customer_count,customer_ratio
0,M,80,53.33
1,F,70,46.67


In [33]:
product_category = products['category'].value_counts(dropna=False).reset_index()
product_category.columns = ['category', 'product_count']
product_category['product_ratio'] = (
    product_category['product_count'] / product_category['product_count'].sum() * 100
).round(2)

product_category

,category,product_count,product_ratio
0,패션,16,16.67
1,뷰티,16,16.67
2,식품,16,16.67
3,도서,13,13.54
4,전자기기,13,13.54
5,스포츠,10,10.42
6,생활용품,9,9.38
7,전자 기기,1,1.04
8,생활 용품,1,1.04
9,뷰 티,1,1.04


In [34]:
products['price'].describe()

count          96.00
mean      204,895.82
std     1,011,847.94
min         5,000.00
25%        52,750.00
50%        94,500.00
75%       153,250.00
max     9,999,999.00
Name: price, dtype: float64

In [35]:
category_price = (
    products
    .groupby('category', as_index=False)
    .agg(
        product_count=('product_id', 'count'),
        avg_price=('price', 'mean'),
        min_price=('price', 'min'),
        max_price=('price', 'max'),
    )
    .assign(avg_price=lambda df: df['avg_price'].round(0))
    .sort_values('avg_price', ascending=False)
)

category_price

,category,product_count,avg_price,min_price,max_price
2,뷰티,16,"710,937.00","18,000.00","9,999,999.00"
3,생활 용품,1,"124,000.00","124,000.00","124,000.00"
6,식품,16,"120,812.00","11,000.00","195,000.00"
8,전자기기,13,"120,231.00","38,000.00","178,000.00"
9,패션,16,"115,125.00","28,000.00","197,000.00"
0,도서,13,"103,692.00","34,000.00","182,000.00"
5,스포츠,10,"78,400.00","5,000.00","183,000.00"
4,생활용품,9,"69,556.00","29,000.00","147,000.00"
1,뷰 티,1,"54,000.00","54,000.00","54,000.00"
7,전자 기기,1,"21,000.00","21,000.00","21,000.00"


In [36]:
order_status = orders['order_status'].value_counts(dropna=False).reset_index()
order_status.columns = ['order_status', 'order_count']
order_status['order_ratio'] = (
    order_status['order_count'] / order_status['order_count'].sum() * 100
).round(2)

order_status

,order_status,order_count,order_ratio
0,completed,180,60.00
1,cancelled,63,21.00
2,refunded,57,19.00


In [37]:
payment_method = orders['payment_method'].value_counts(dropna=False).reset_index()
payment_method.columns = ['payment_method', 'order_count']
payment_method['order_ratio'] = (
    payment_method['order_count'] / payment_method['order_count'].sum() * 100
).round(2)

payment_method

,payment_method,order_count,order_ratio
0,kakao_pay,90,30.00
1,card,72,24.00
2,bank_transfer,70,23.33
3,naver_pay,68,22.67


In [38]:
# 완료 주문 범위 확정과 병합 점검
items_with_orders = order_items.merge(
    orders[['order_id', 'customer_id', 'order_date', 'order_month', 'order_status', 'payment_method']],
    on='order_id', how='left', validate='many_to_one', indicator='order_match')
assert len(items_with_orders) == len(order_items)
completed = items_with_orders.loc[items_with_orders['order_status'].eq('completed')].copy()
sales_items = completed.merge(
    products[['product_id', 'category']], on='product_id', how='left',
    validate='many_to_one', indicator='product_match')
sales_items = sales_items.merge(
    customers[['customer_id', 'city']], on='customer_id', how='left',
    validate='many_to_one', indicator='customer_match')
assert len(sales_items) == len(completed)
assert sales_items['order_item_id'].is_unique
sales_items['category'] = sales_items['category'].fillna('카테고리 미확인')
sales_items['city'] = sales_items['city'].fillna('고객 미연결')
merge_check = pd.DataFrame([
    ['주문 연결 · 전체 상세', len(order_items), len(items_with_orders), int(items_with_orders['order_match'].eq('left_only').sum())],
    ['상품 연결 · completed', len(completed), len(sales_items), int(sales_items['product_match'].eq('left_only').sum())],
    ['고객 연결 · completed', len(completed), len(sales_items), int(sales_items['customer_match'].eq('left_only').sum())],
], columns=['연결', '병합 전', '병합 후', '미연결 상세 행'])
display(merge_check)

completed_ids = orders.loc[orders['order_status'].eq('completed'), 'order_id']
no_detail = completed_ids.loc[~completed_ids.isin(sales_items['order_id'])]
quality_scope = pd.DataFrame([
    ['주문 상태 미확인', int(items_with_orders['order_match'].eq('left_only').sum()), float(items_with_orders.loc[items_with_orders['order_match'].eq('left_only'), 'line_total'].sum()), 'completed 판별 불가: 제외'],
    ['카테고리 미확인', int(sales_items['product_match'].eq('left_only').sum()), float(sales_items.loc[sales_items['product_match'].eq('left_only'), 'line_total'].sum()), 'completed 총합에 포함'],
    ['월 미확인', int(sales_items['order_date'].isna().sum()), float(sales_items.loc[sales_items['order_date'].isna(), 'line_total'].sum()), 'completed 총합에 포함'],
    ['고객 미연결', int(sales_items['customer_match'].eq('left_only').sum()), float(sales_items.loc[sales_items['customer_match'].eq('left_only'), 'line_total'].sum()), '총합 포함, 고객 특성 비교 제외'],
], columns=['확인 사항', '상세 행', '금액', '처리 기준'])
display(quality_scope)
print('주문표 completed:', len(completed_ids), '/ 상세 존재:', sales_items['order_id'].nunique(), '/ 상세 없음:', len(no_detail))
print('분석 상세:', len(sales_items), '/ 금액:', f"{sales_items['line_total'].sum():,.0f}")

,연결,병합 전,병합 후,미연결 상세 행
0,주문 연결 · 전체 상세,752,752,2
1,상품 연결 · completed,446,446,16
2,고객 연결 · completed,446,446,1


,확인 사항,상세 행,금액,처리 기준
0,주문 상태 미확인,2,"922,000.00",completed 판별 불가: 제외
1,카테고리 미확인,16,"4,015,000.00",completed 총합에 포함
2,월 미확인,6,"1,903,000.00",completed 총합에 포함
3,고객 미연결,1,"376,000.00","총합 포함, 고객 특성 비교 제외"


주문표 completed: 180 / 상세 존재: 179 / 상세 없음: 1
분석 상세: 446 / 금액: 141,340,000


주문 상태 미확인 상세 2행은 completed 여부를 판단할 수 없어 제외했다. 상품·고객·날짜 미확인은 별도 범주로 보존했다. 확인 항목은 서로 겹칠 수 있으므로 미확인 금액끼리 더하지 않는다. 상세가 없는 완료 주문 1건도 금액 0으로 간주하지 않는다.

In [39]:
# 카테고리별 completed 금액과 수량
category_sales = (
    sales_items.groupby('category', as_index=False, dropna=False)
    .agg(total_quantity=('quantity', 'sum'), total_sales=('line_total', 'sum'),
         order_count=('order_id', 'nunique'))
    .sort_values(['total_sales', 'category'], ascending=[False, True]).reset_index(drop=True))
category_sales['sales_ratio'] = (category_sales['total_sales'] / sales_items['line_total'].sum() * 100).round(2)
category_sales['weighted_unit_price'] = (category_sales['total_sales'] / category_sales['total_quantity']).round(2)
display(category_sales)
print('금액 1위:', category_sales.iloc[0]['category'])

,category,total_quantity,total_sales,order_count,sales_ratio,weighted_unit_price
0,패션,290.00,"32,940,000.00",68,23.31,"113,586.21"
1,식품,226.00,"25,035,000.00",63,17.71,"110,774.34"
2,전자기기,176.00,"21,485,000.00",54,15.20,"122,073.86"
3,도서,174.00,"17,663,000.00",49,12.50,"101,511.49"
4,뷰티,168.00,"17,103,000.00",54,12.10,"101,803.57"
5,스포츠,165.00,"13,237,000.00",47,9.37,"80,224.24"
6,생활용품,108.00,"7,001,000.00",37,4.95,"64,824.07"
7,카테고리 미확인,34.00,"4,015,000.00",16,2.84,"118,088.24"
8,생활 용품,17.00,"2,108,000.00",5,1.49,"124,000.00"
9,뷰 티,12.00,"648,000.00",4,0.46,"54,000.00"


금액 1위: 패션


In [40]:
# 상품 구성과 주문 상세의 단가를 구분해 비교
category_sales_with_price = category_sales.merge(
    category_price[['category', 'product_count', 'avg_price']],
    on='category', how='left', validate='one_to_one')
display(category_sales_with_price[['category', 'product_count', 'avg_price', 'weighted_unit_price', 'total_sales']])
print('avg_price: 상품표의 단순 평균 가격 / weighted_unit_price: 실제 주문 상세 금액 ÷ 수량')

,category,product_count,avg_price,weighted_unit_price,total_sales
0,패션,16.00,"115,125.00","113,586.21","32,940,000.00"
1,식품,16.00,"120,812.00","110,774.34","25,035,000.00"
2,전자기기,13.00,"120,231.00","122,073.86","21,485,000.00"
3,도서,13.00,"103,692.00","101,511.49","17,663,000.00"
4,뷰티,16.00,"710,937.00","101,803.57","17,103,000.00"
5,스포츠,10.00,"78,400.00","80,224.24","13,237,000.00"
6,생활용품,9.00,"69,556.00","64,824.07","7,001,000.00"
7,카테고리 미확인,NaN,NaN,"118,088.24","4,015,000.00"
8,생활 용품,1.00,"124,000.00","124,000.00","2,108,000.00"
9,뷰 티,1.00,"54,000.00","54,000.00","648,000.00"


avg_price: 상품표의 단순 평균 가격 / weighted_unit_price: 실제 주문 상세 금액 ÷ 수량


In [41]:
# 월별 금액·고유 주문 수·주문당 평균 금액
order_sales = sales_items.copy()
monthly_sales = (
    order_sales.groupby('order_month', as_index=False, dropna=False)
    .agg(total_sales=('line_total', 'sum'), order_count=('order_id', 'nunique'))
    .sort_values('order_month').reset_index(drop=True))
monthly_sales['avg_order_value'] = (monthly_sales['total_sales'] / monthly_sales['order_count']).round(2)
display(monthly_sales)
print('월 미확인도 합계에 포함. 해당 행은 실제 달로 해석하지 않는다.')

,order_month,total_sales,order_count,avg_order_value
0,2025-07,"9,561,000.00",11,"869,181.82"
1,2025-08,"9,362,000.00",17,"550,705.88"
2,2025-09,"5,299,000.00",4,"1,324,750.00"
3,2025-10,"12,629,000.00",17,"742,882.35"
4,2025-11,"9,583,000.00",16,"598,937.50"
5,2025-12,"8,581,000.00",12,"715,083.33"
6,2026-01,"8,135,000.00",9,"903,888.89"
7,2026-02,"7,689,000.00",12,"640,750.00"
8,2026-03,"7,247,000.00",13,"557,461.54"
9,2026-04,"11,459,000.00",14,"818,500.00"


월 미확인도 합계에 포함. 해당 행은 실제 달로 해석하지 않는다.


In [42]:
# 고객별 구매 금액과 횟수: 이름은 사용하지 않는다
customer_sales = (
    order_sales.groupby(['customer_id', 'city'], as_index=False, dropna=False)
    .agg(order_count=('order_id', 'nunique'), total_sales=('line_total', 'sum'))
    .sort_values(['total_sales', 'customer_id'], ascending=[False, True]).reset_index(drop=True))
customer_sales['avg_order_value'] = (customer_sales['total_sales'] / customer_sales['order_count']).round(2)
display(customer_sales[['customer_id', 'order_count', 'total_sales', 'avg_order_value']].head(10))
known_customers = customer_sales.loc[customer_sales['customer_id'].isin(customers['customer_id'])].copy()
known_customers['purchase_group'] = np.where(known_customers['order_count'].ge(2), '2건 이상', '1건')
repeat_summary = known_customers.groupby('purchase_group', as_index=False).agg(
    customer_count=('customer_id', 'nunique'), order_count=('order_count', 'sum'), total_sales=('total_sales', 'sum'))
repeat_summary['sales_ratio'] = (repeat_summary['total_sales'] / known_customers['total_sales'].sum() * 100).round(2)
display(repeat_summary)
print('고객표와 연결된 구매 고객:', len(known_customers), '/ 미연결 ID:', len(customer_sales) - len(known_customers))

,customer_id,order_count,total_sales,avg_order_value
0,3.00,3,"6,702,000.00","2,234,000.00"
1,14.00,4,"5,204,000.00","1,301,000.00"
2,90.00,3,"4,311,000.00","1,437,000.00"
3,93.00,4,"4,001,000.00","1,000,250.00"
4,65.00,6,"3,977,000.00","662,833.33"
5,104.00,2,"3,966,000.00","1,983,000.00"
6,20.00,4,"3,808,000.00","952,000.00"
7,38.00,4,"3,656,000.00","914,000.00"
8,28.00,4,"3,201,000.00","800,250.00"
9,22.00,3,"3,109,000.00","1,036,333.33"


,purchase_group,customer_count,order_count,total_sales,sales_ratio
0,1건,45,45,"32,224,000.00",22.86
1,2건 이상,53,133,"108,740,000.00",77.14


고객표와 연결된 구매 고객: 98 / 미연결 ID: 1


In [43]:
# 원본에서 독립적으로 완료 주문 상세를 선택해 총합 검증
original_completed = raw_items.loc[raw_items['order_id'].isin(completed_ids)].copy()
original_total = (original_completed['quantity'] * original_completed['unit_price']).sum()
totals = pd.DataFrame({
    '구분': ['원본 completed', '카테고리별', '월별', '고객별'],
    'total_sales': [original_total, category_sales['total_sales'].sum(), monthly_sales['total_sales'].sum(), customer_sales['total_sales'].sum()]})
totals['원본과 차이'] = totals['total_sales'] - original_total
totals['일치'] = totals['원본과 차이'].eq(0)
assert totals['일치'].all()
assert len(original_completed) == len(sales_items)
assert set(original_completed['order_item_id']) == set(sales_items['order_item_id'])
assert monthly_sales['order_count'].sum() == customer_sales['order_count'].sum() == sales_items['order_id'].nunique()
assert category_sales['total_quantity'].sum() == original_completed['quantity'].sum()
assert sales_items['order_status'].eq('completed').all()
display(totals)
print('PASS: 4개 금액 합계·상세 ID·고유 주문 수 일치')
print('카테고리 미확인을 빼면:', f"{category_sales.loc[category_sales['category'].ne('카테고리 미확인'), 'total_sales'].sum():,.0f}")
print('월 미확인을 빼면:', f"{monthly_sales.loc[monthly_sales['order_month'].ne('월 미확인'), 'total_sales'].sum():,.0f}")
print('총합 일치는 남아 있는 품질 문제가 해결됐다는 뜻은 아니다.')

,구분,total_sales,원본과 차이,일치
0,원본 completed,"141,340,000.00",0.00,True
1,카테고리별,"141,340,000.00",0.00,True
2,월별,"141,340,000.00",0.00,True
3,고객별,"141,340,000.00",0.00,True


PASS: 4개 금액 합계·상세 ID·고유 주문 수 일치
카테고리 미확인을 빼면: 137,325,000
월 미확인을 빼면: 139,437,000
총합 일치는 남아 있는 품질 문제가 해결됐다는 뜻은 아니다.


![카테고리별 completed 금액과 수량](images/step02_eda.png)

![원본과 집계 결과의 총합 검증](images/step03_totals.png)

### 결과 관찰

패션의 completed 금액은 32,940,000으로 전체의 23.31%이며 가장 크다. 수량도 290개로 가장 많다. 상품 수는 패션·뷰티·식품이 각각 16개지만 금액은 다르다. ‘뷰 티’, ‘생활 용품’, ‘전자 기기’처럼 분리된 표기도 남아 있다.

날짜가 확인된 달 중 2026년 8월 금액이 16,980,000으로 가장 크다. 주문 15건, 주문당 평균 금액 1,132,000이다. 반면 주문 수가 가장 많은 달은 2025년 8월·10월로 각각 17건이다.

고객표와 연결된 구매 고객 98명 중 2건 이상 구매한 고객은 53명이다. 이들의 금액은 108,740,000이다. 고객 미연결 금액 376,000은 총합에는 포함하고 이 비교에서는 제외했다.

### 나의 해석과 판단

금액 순위는 상품 수만으로 설명되지 않는다. 수량과 실제 주문 단가를 함께 봐야 한다. 월별 금액도 주문 수와 주문당 평균 금액을 나누어 확인해야 한다. 2건 이상 구매는 이번 자료의 관측 기간 내 기준이며 충성 고객이나 장기 유지 고객을 뜻하지 않는다.

### 업무·분석적 의미

상품별 판매 구성, 월별 주문 규모와 객단가, 반복 구매 집단을 다음 분석 대상으로 좁힐 수 있다. 특히 미연결·날짜 누락 금액을 따로 기록하면 어느 범위까지 집계 결과를 활용할 수 있는지 설명할 수 있다.

### 한계와 추가 확인 사항

전처리 후에도 날짜 누락, 상품·고객 미연결, 극단값과 카테고리 표기 차이가 남았다. 고객 나이 최댓값은 150, 상품 가격 최댓값은 9,999,999다. 상품 평균 가격을 대표적인 구매 단가로 해석하기 어렵다.

작성일 이후인 2027년 주문도 completed 집계에 남아 있다. 날짜가 변환됐다고 사실에 맞는 날짜라고 단정할 수 없어 원자료 확인이 필요하다. 월별 표는 관측된 달만 보여 주며, 기록이 없는 달을 영업 실적 0으로 단정하지 않는다. 자료 범위·기간과 실제 원가·프로모션 정보가 없어 계절성, 인기, 수익성이나 인과관계를 결론 내리지 않는다.

공식 Notebook의 추가 실습도 같은 completed 범위에서 확인했다. 도시별 고객 수는 고객표 전체, 구매 금액은 완료 주문에 근거하므로 서로 다른 분모임을 구분한다.

In [44]:
# 도시별 고객 분포와 completed 금액
city_amount = sales_items.groupby('city', as_index=False, dropna=False).agg(
    purchasing_ids=('customer_id', 'nunique'), order_count=('order_id', 'nunique'), total_sales=('line_total', 'sum'))
city_comparison = customer_city[['city', 'customer_count']].merge(city_amount, on='city', how='outer', validate='one_to_one')
city_comparison['customer_count'] = city_comparison['customer_count'].fillna(0).astype(int)
city_comparison[['purchasing_ids', 'order_count', 'total_sales']] = city_comparison[['purchasing_ids', 'order_count', 'total_sales']].fillna(0)
assert city_comparison['total_sales'].sum() == original_total
assert city_comparison['customer_count'].sum() == len(customers)
display(city_comparison.sort_values('total_sales', ascending=False).reset_index(drop=True))

,city,customer_count,purchasing_ids,order_count,total_sales
0,부산,23,16,32,"24,916,000.00"
1,서울,18,11,20,"22,211,000.00"
2,광주,21,16,28,"19,866,000.00"
3,인천,16,12,23,"15,723,000.00"
4,대구,21,12,18,"14,781,000.00"
5,수원,10,8,13,"12,220,000.00"
6,울산,17,9,17,"11,545,000.00"
7,성남,9,5,13,"11,288,000.00"
8,고양,8,6,10,"7,222,000.00"
9,Unknown,2,1,1,"820,000.00"


In [45]:
# 결제수단별 평균 주문 금액
payment_sales = sales_items.groupby('payment_method', as_index=False, dropna=False).agg(
    total_sales=('line_total', 'sum'), order_count=('order_id', 'nunique'))
payment_sales['avg_order_value'] = (payment_sales['total_sales'] / payment_sales['order_count']).round(2)
assert payment_sales['total_sales'].sum() == original_total
assert payment_sales['order_count'].sum() == sales_items['order_id'].nunique()
display(payment_sales.sort_values('avg_order_value', ascending=False).reset_index(drop=True))

,payment_method,total_sales,order_count,avg_order_value
0,card,"37,542,000.00",40,"938,550.00"
1,bank_transfer,"39,461,000.00",49,"805,326.53"
2,kakao_pay,"36,861,000.00",50,"737,220.00"
3,naver_pay,"27,476,000.00",40,"686,900.00"


In [46]:
# 월별·카테고리별 금액 교차 집계
month_category = sales_items.pivot_table(
    index='order_month', columns='category', values='line_total', aggfunc='sum', fill_value=0)
assert month_category.to_numpy().sum() == original_total
pd.testing.assert_series_equal(month_category.sum(axis=1), monthly_sales.set_index('order_month')['total_sales'], check_names=False)
display(month_category)

category,도서,뷰 티,뷰티,생활 용품,생활용품,스포츠,식품,전자 기기,전자기기,카테고리 미확인,패션
order_month,,,,,,,,,,,
2025-07,"1,040,000.00",0.00,"1,133,000.00",0.00,"406,000.00","295,000.00","2,123,000.00",0.00,"1,319,000.00","82,000.00","3,163,000.00"
2025-08,"1,144,000.00",0.00,"1,050,000.00",0.00,"983,000.00","417,000.00","505,000.00",0.00,"2,547,000.00","470,000.00","2,246,000.00"
2025-09,"380,000.00",0.00,"125,000.00",0.00,"735,000.00","438,000.00","528,000.00",0.00,"1,157,000.00","780,000.00","1,156,000.00"
2025-10,"1,570,000.00",0.00,"661,000.00",0.00,"1,334,000.00","1,426,000.00","3,088,000.00",0.00,"1,781,000.00","82,000.00","2,687,000.00"
2025-11,"583,000.00","108,000.00","1,473,000.00",0.00,"528,000.00","1,271,000.00","2,221,000.00",0.00,"1,035,000.00","420,000.00","1,944,000.00"
2025-12,"34,000.00",0.00,"884,000.00",0.00,"489,000.00","791,000.00","2,048,000.00",0.00,"1,408,000.00","468,000.00","2,459,000.00"
2026-01,"74,000.00",0.00,"376,000.00",0.00,"804,000.00","2,359,000.00","2,465,000.00","105,000.00","485,000.00",0.00,"1,467,000.00"
2026-02,"1,827,000.00",0.00,"275,000.00","124,000.00","522,000.00","1,009,000.00","1,193,000.00",0.00,"1,562,000.00",0.00,"1,177,000.00"
2026-03,"1,075,000.00",0.00,"728,000.00",0.00,"233,000.00","364,000.00","2,792,000.00",0.00,"1,378,000.00","140,000.00","537,000.00"


도시·결제수단·월×카테고리 금액도 원본 completed 합계와 일치했다. 지역별 기록 수는 전체 시장 점유율을 뜻하지 않으며, 결제수단별 객단가 차이도 수단 자체의 효과로 볼 수 없다.

## 3. 관찰 → 가설 → 추가 검증

### 사례 1
- 관찰: 패션의 금액은 32,940,000, 수량은 290개로 각각 가장 크다. 상품 수는 패션·뷰티·식품이 16개로 같다.
- 가설: 상품 수보다 판매 수량과 주문 단가 구성이 금액 차이와 연결될 수 있다. 대량 주문이 일부 영향을 줄 가능성도 있다.
- 추가 검증: 카테고리별 수량 가중 단가와 주문당 금액 분포를 비교한다. 패션의 수량 50개 주문 상세가 포함·제외된 경우를 나누어 순위가 달라지는지 확인한다. 제외 분석은 오류 확정이나 원본 삭제를 뜻하지 않는다.

### 사례 2
- 관찰: 2026년 8월의 금액은 16,980,000, 주문은 15건이다. 주문 수 최다 달의 17건보다 적다.
- 가설: 금액이 큰 달에는 주문당 금액이나 구매 카테고리 구성이 다를 수 있다.
- 추가 검증: 월별 주문 수×주문당 평균 금액을 대조하고, 2026년 8월과 전월의 월×카테고리 금액 및 고액 주문을 비교한다. 월 미확인·미래 날짜와 관측 기간을 먼저 점검한다.

### 사례 3
- 관찰: 고객표와 연결된 구매 고객 중 2건 이상 구매 집단은 53명, 금액은 108,740,000이다. 1건 집단은 45명, 금액은 32,224,000이다.
- 가설: 여러 번 구매하는 고객 집단의 누적 금액이 큰 편일 수 있지만, 고객 수와 관측 기간 차이도 작용할 수 있다.
- 추가 검증: 고객당 금액과 주문당 금액을 구분하고 가입 시점·관측 기간을 맞춘다. 1건 구매 고객 중 고액 구매 집단도 따로 확인한다. 현재 기록만으로 충성도나 미래 이탈을 판단하지 않는다.

In [47]:
# 반복 구매와 고액 단발 구매를 구분하는 탐색 기준
single = known_customers.loc[known_customers['order_count'].eq(1)].copy()
single_threshold = single['total_sales'].quantile(0.75)
single['purchase_type'] = np.where(single['total_sales'].ge(single_threshold), '1건 · 상위 25% 금액 이상', '1건 · 그 외')
single_summary = single.groupby('purchase_type', as_index=False).agg(
    customer_count=('customer_id', 'nunique'), total_sales=('total_sales', 'sum'))
display(single_summary)
print('1건 구매 집단의 75% 분위수:', f'{single_threshold:,.0f}')
print('분위수 경계 동률이 있으면 인원 비율은 정확히 25%가 아닐 수 있다. 업무 기준으로 확정하지 않는다.')

,purchase_type,customer_count,total_sales
0,1건 · 그 외,33,"16,447,000.00"
1,1건 · 상위 25% 금액 이상,12,"15,777,000.00"


1건 구매 집단의 75% 분위수: 1,007,000
분위수 경계 동률이 있으면 인원 비율은 정확히 25%가 아닐 수 있다. 업무 기준으로 확정하지 않는다.


## 4. LLM 질문 확장

- Safe Context: 개인정보 원본 대신 표 구조, completed 범위, 카테고리·월·구매 횟수 집계, 품질 점검 결과만 사용했다.
- Prompt: 아래 구조·집계 요약을 바탕으로 현재 데이터로 계산 가능한 후속 질문을 제안해 달라고 Codex에 요청했다. 원인 단정을 금지하고 필요한 컬럼·검증 방법·한계를 구분하도록 했다.
- LLM 제안 요약: ① 카테고리별 수량·단가 분해 ② 월별 주문 수·객단가 및 카테고리 구성 비교 ③ 반복 구매·고액 단발 구매 분리 ④ 마케팅 효과는 현재 변수로 확인할 수 없어 월별 구성 비교로 수정.
- 실제 사용 여부: 수정 후 사용. 계산 가능한 비교만 선택하고, 원인 설명은 가설로 남겼다.
- 판단 이유: 수량·단가·주문일·상태·식별자가 있어 집계할 수 있지만, 원가·프로모션·방문 기록은 없다. 미확인 범주를 누락시키면 총합이 달라진다.
- 사람이 수정한 내용: 실행 결과와 LLM 수정안을 검토해 미확인 카테고리·월도 총합에 포함하는 기준을 채택했다. 금액 순위를 인기·수익성으로 해석하지 않고, 마케팅 효과는 관련 자료가 없어 판단을 보류했다. 표기 통합과 고액 상세 제외는 별도 비교로만 확인하고 본 집계는 유지했다.

In [48]:
# 집계 정보만 사용한 질문 확장 요청
safe_context = {
    '데이터': {name: [col for col in frame.columns if col != 'name'] for name, frame in data.items()},
    '범위': {'상태': 'completed', '상세 행': len(sales_items), '고유 주문': int(sales_items['order_id'].nunique()), '금액 합계': int(original_total)},
    '카테고리': category_sales[['category', 'total_quantity', 'total_sales']].to_dict('records'),
    '월별': monthly_sales.to_dict('records'),
    '고객 집단': repeat_summary.to_dict('records'),
    '품질': quality_scope.to_dict('records'),
}
prompt = """온라인 쇼핑몰 실습 데이터의 구조와 집계 요약이다.
현재 데이터로 답할 수 있는 후속 EDA 질문 4개를 제안해 줘.
각 질문의 필요한 컬럼, 계산 지표, 검증 방법, 한계를 구분해 줘.
카테고리 금액 차이는 수량·단가로, 월별 변화는 주문 수·주문당 금액으로 나누어 봐 줘.
반복 구매와 고액 단발 구매를 구분해 줘.
마케팅 효과처럼 현재 자료로 답할 수 없는 질문은 계산 가능한 질문으로 바꿔 줘.
개인정보 원본을 요청하지 말고, 데이터에 없는 원인이나 인과관계를 단정하지 마."""
print(prompt)
print('제공 범위:', safe_context['범위'])
print('제공 내용: 컬럼 구조, 카테고리·월별·고객 집단 집계, 미연결·날짜 누락 요약')
print('고객 이름과 개인별 거래 내역은 제공하지 않았다.')

온라인 쇼핑몰 실습 데이터의 구조와 집계 요약이다.
현재 데이터로 답할 수 있는 후속 EDA 질문 4개를 제안해 줘.
각 질문의 필요한 컬럼, 계산 지표, 검증 방법, 한계를 구분해 줘.
카테고리 금액 차이는 수량·단가로, 월별 변화는 주문 수·주문당 금액으로 나누어 봐 줘.
반복 구매와 고액 단발 구매를 구분해 줘.
마케팅 효과처럼 현재 자료로 답할 수 없는 질문은 계산 가능한 질문으로 바꿔 줘.
개인정보 원본을 요청하지 말고, 데이터에 없는 원인이나 인과관계를 단정하지 마.
제공 범위: {'상태': 'completed', '상세 행': 446, '고유 주문': 179, '금액 합계': 141340000}
제공 내용: 컬럼 구조, 카테고리·월별·고객 집단 집계, 미연결·날짜 누락 요약
고객 이름과 개인별 거래 내역은 제공하지 않았다.


In [49]:
# LLM 제안의 계산 가능성과 판단
llm_review = pd.DataFrame([
    ['수량·단가로 카테고리 차이 확인', 'quantity, unit_price, category', '금액÷수량, 고액 상세 민감도', '수정 후 사용', '상품 평균 가격 대신 주문 단가 사용'],
    ['월별 주문 수·객단가 비교', 'order_date, order_id, line_total', '금액÷고유 주문 수, 월×카테고리', '수정 후 사용', '미확인 월·미래 날짜를 별도 확인'],
    ['반복 구매와 고액 단발 구매 분리', 'customer_id, order_id, line_total', '1건/2건 이상, 단발 집단 분위수', '사용', '고객 미연결 제외, 관측 기간 제한'],
    ['마케팅이 금액을 늘렸는가?', '프로모션·노출 기록 없음', '효과 추정 대신 월별 구성 비교', '보류', '현재 자료로 인과 효과를 식별할 수 없음'],
], columns=['LLM 제안', '필요한 데이터', '계산·검증', '실제 사용 여부', '판단 이유'])
display(llm_review)

,LLM 제안,필요한 데이터,계산·검증,실제 사용 여부,판단 이유
0,수량·단가로 카테고리 차이 확인,"quantity, unit_price, category","금액÷수량, 고액 상세 민감도",수정 후 사용,상품 평균 가격 대신 주문 단가 사용
1,월별 주문 수·객단가 비교,"order_date, order_id, line_total","금액÷고유 주문 수, 월×카테고리",수정 후 사용,미확인 월·미래 날짜를 별도 확인
2,반복 구매와 고액 단발 구매 분리,"customer_id, order_id, line_total","1건/2건 이상, 단발 집단 분위수",사용,"고객 미연결 제외, 관측 기간 제한"
3,마케팅이 금액을 늘렸는가?,프로모션·노출 기록 없음,효과 추정 대신 월별 구성 비교,보류,현재 자료로 인과 효과를 식별할 수 없음


In [50]:
# 표기 통합과 고액 상세의 영향 확인: CSV 원본은 유지
category_alias = {'전자 기기': '전자기기', '생활 용품': '생활용품', '뷰 티': '뷰티'}
normalized = sales_items.assign(category=sales_items['category'].replace(category_alias))
normalized_category = normalized.groupby('category', as_index=False).agg(total_sales=('line_total', 'sum'))
assert normalized_category['total_sales'].sum() == original_total
print('표기 통합 전후 총합 일치:', normalized_category['total_sales'].sum() == original_total)
print('표기 통합 전/후 카테고리 수:', sales_items['category'].nunique(), '/', normalized['category'].nunique())
print('표기 통합 후 금액 1위:', normalized_category.sort_values('total_sales', ascending=False).iloc[0]['category'])
fashion_large = sales_items['category'].eq('패션') & sales_items['quantity'].eq(50)
without_large = sales_items.loc[~fashion_large].groupby('category', as_index=False).agg(total_sales=('line_total', 'sum'))
print('패션 수량 50개 상세:', int(fashion_large.sum()), '행 / 금액:', f"{sales_items.loc[fashion_large, 'line_total'].sum():,.0f}")
print('해당 상세 제외 시 패션 금액:', f"{without_large.loc[without_large['category'].eq('패션'), 'total_sales'].sum():,.0f}")
print('제외 시 금액 1위:', without_large.sort_values('total_sales', ascending=False).iloc[0]['category'])

표기 통합 전후 총합 일치: True
표기 통합 전/후 카테고리 수: 11 / 8
표기 통합 후 금액 1위: 패션
패션 수량 50개 상세: 1 행 / 금액: 4,800,000
해당 상세 제외 시 패션 금액: 28,140,000
제외 시 금액 1위: 패션


표기 3개를 통합하면 미확인 범주를 포함한 카테고리는 11개에서 8개로 줄지만 총합과 금액 1위는 유지됐다. 패션의 수량 50개 상세 금액 4,800,000을 제외한 민감도 분석에서도 패션 금액 28,140,000이 가장 크다. 한 건만으로 순위가 결정된 것은 아니지만, 그 주문이 정상인지와 판매량 차이의 원인은 확인하지 못했다. 본 집계에서는 원본 표기와 해당 상세를 유지했다.

![LLM 제안의 계산 가능성과 판단](images/step04_llm.png)

In [51]:
# 공식 EDA 집계와 분석 범위를 대조
# 공식 src/eda.py의 집계 방식을 같은 입력으로 재현한다.
official_items = order_items.merge(products[['product_id', 'category']], on='product_id', how='left')
official_category = official_items.groupby('category', as_index=False).agg(total_sales=('line_total', 'sum'))
official_order_sales = order_items.merge(orders, on='order_id', how='left')
official_order_sales['order_date'] = pd.to_datetime(official_order_sales['order_date'], errors='coerce')
official_order_sales['order_month'] = official_order_sales['order_date'].dt.to_period('M').astype(str)
official_monthly = official_order_sales.groupby('order_month', as_index=False).agg(total_sales=('line_total', 'sum'))
scope_comparison = pd.DataFrame([
    ['공식 예제 방식 · 카테고리', official_category['total_sales'].sum(), '상태 필터 없음, 미확인 카테고리 누락'],
    ['공식 예제 방식 · 월별', official_monthly['total_sales'].sum(), '상태 필터 없음, 날짜 누락 처리 차이'],
    ['이번 답안 · 원본 completed', original_total, '미확인 카테고리·월 포함'],
    ['이번 답안 · 카테고리', category_sales['total_sales'].sum(), '원본 completed와 일치'],
    ['이번 답안 · 월별', monthly_sales['total_sales'].sum(), '원본 completed와 일치'],
], columns=['비교 대상', '금액', '범위 판단'])
display(scope_comparison)
print('공식 보조 스크립트의 집계도 같은 필터·결측 처리 조건으로 맞춘 뒤 비교해야 한다.')

,비교 대상,금액,범위 판단
0,공식 예제 방식 · 카테고리,"245,916,000.00","상태 필터 없음, 미확인 카테고리 누락"
1,공식 예제 방식 · 월별,"248,630,000.00","상태 필터 없음, 날짜 누락 처리 차이"
2,이번 답안 · 원본 completed,"141,340,000.00",미확인 카테고리·월 포함
3,이번 답안 · 카테고리,"141,340,000.00",원본 completed와 일치
4,이번 답안 · 월별,"141,340,000.00",원본 completed와 일치


공식 보조 스크립트의 집계도 같은 필터·결측 처리 조건으로 맞춘 뒤 비교해야 한다.


In [52]:
# 집계 저장과 CSV 재로딩 검증
eda_result_summary = pd.DataFrame([
    ['카테고리별 금액·수량', '수량과 주문 단가를 함께 비교', 'category_sales'],
    ['월별 금액·주문 수', '월×카테고리 구성과 고액 주문 확인', 'monthly_sales'],
    ['반복 구매와 단발 구매', '고객별 관측 기간을 맞춰 비교', 'customer_sales'],
], columns=['질문', '다음 검증', '결과표'])
result_tables = {'category_sales': category_sales, 'monthly_sales': monthly_sales,
                 'customer_sales': customer_sales, 'customer_city': customer_city,
                 'product_category': product_category, 'eda_questions': eda_result_summary,
                 'validation': totals, 'llm_review': llm_review}
reload_checks = []
for name, table in result_tables.items():
    path = REPORT_DIR / f'ch06_{name}.csv'
    table.to_csv(path, index=False, encoding='utf-8-sig')
    reloaded = pd.read_csv(path)
    pd.testing.assert_frame_equal(table.reset_index(drop=True), reloaded, check_dtype=False)
    reload_checks.append({'결과표': name, 'CSV 재로딩 일치': True})
after_hashes = {name: hashlib.sha256((PROCESSED_DIR / name).read_bytes()).hexdigest() for name in filenames}
assert input_hashes == after_hashes
display(pd.DataFrame(reload_checks))
print('입력 CSV 4개 보존:', input_hashes == after_hashes)

,결과표,CSV 재로딩 일치
0,category_sales,True
1,monthly_sales,True
2,customer_sales,True
3,customer_city,True
4,product_category,True
5,eda_questions,True
6,validation,True
7,llm_review,True


입력 CSV 4개 보존: True


## 5. 다음 분석 우선순위
1. 상품·고객 미연결, 상세가 없는 완료 주문, 날짜 누락과 작성일 이후 날짜의 원인을 확인한다. 오류 정정 가능 여부와 분석에서 유지·제외할 기준을 먼저 정한다.
2. 카테고리 표기를 확인한 뒤 수량·주문 단가·고액 주문의 영향을 나누어 비교한다. 금액 순위가 집계 기준에 따라 바뀌는지 점검한다.
3. 날짜와 분석 기간을 정리한 뒤 월별 주문 수·객단가·카테고리 구성을 비교한다. 고객별 관측 기간도 맞춰 반복 구매와 고액 단발 구매를 구분한다.

### 가장 먼저 확인하고 싶은 이유

같은 completed 상태라도 전처리 후 남아 있는 상세와 연결 상태에 따라 분석 범위가 달라진다. 현재 총합은 일치하지만, 누락된 거래의 실제 금액이나 잘못된 날짜가 복원된 것은 아니다. 이를 먼저 확인해야 상품 운영과 반복 구매 관리에 사용할 비교 기준이 흔들리지 않는다.

### 현재 데이터만으로 단정할 수 없는 것

높은 금액이 높은 인기·이익을 의미하는지, 월별 변화가 마케팅이나 계절성 때문인지, 여러 번 구매한 고객이 장기적으로 유지되는지는 단정할 수 없다. 실습용 자료의 집계 결과를 실제 시장이나 고객 전체로 일반화하지 않는다.

## 최종 체크
- [x] 질문과 지표가 연결되어 있습니다.
- [x] 집계 총합을 검증했습니다.
- [x] 관찰과 가설을 구분했습니다.
- [x] 다음 검증 질문을 작성했습니다.
- [x] LLM 제안을 검증했습니다.
- [ ] 최종 Notebook URL을 제출합니다.